# A1 — Predictive Coding Router (PCR)

Routing skoru: score(i, j) = 1 / (||x_j - f_θ(x_i)||² + δ). Predictor MLP her token için bir sonraki token'ı tahmin eder, tahmin hatası ne kadar büyükse o token'a o kadar çok bakılır.

**Bağımsız çalışır.** Tüm parametreler Hücre 2'de toplanmıştır — sadece o hücreyi düzenleyerek deneyi kontrol edebilirsin. Sonuçlar `./results/A1_PCR_*` altına kaydedilir.

In [ ]:
# ─── HÜCRE 1: Setup ────────────────────────────────────────────────────
# Colab tespit + pip install + Drive mount (opsiyonel) + arf_lib import
import os, sys, importlib
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    # Colab'da arf_lib.py'nin yerini bul — varsayım:
    # repo /content/CALMA/standalone_notebooks/ altında klonlu
    REPO_BASE = "/content/CALMA"
    NB_DIR = f"{REPO_BASE}/standalone_notebooks"
    if not os.path.exists(NB_DIR):
        !git clone https://github.com/hakanskn/CALMA.git {REPO_BASE} 2>&1 | tail -5
    sys.path.insert(0, NB_DIR)

    # Drive mount (sonuçları Drive'a yedeklemek istersen PARAMS'ta results_root değiştir)
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print("Drive mount skipped:", e)

    !pip install -q transformers==4.41.0 datasets==2.19.0 tokenizers==0.19.0 accelerate==0.30.0 matplotlib seaborn
else:
    # Lokal: arf_lib.py notebook ile aynı klasörde
    NB_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    sys.path.insert(0, NB_DIR)

import arf_lib
importlib.reload(arf_lib)
print(f"arf_lib loaded from: {arf_lib.__file__}")
print(f"GPU info: {arf_lib.gpu_info()}")

In [ ]:
# ─── HÜCRE 2: PARAMETRELER ─────────────────────────────────────────────
PARAMS = {
    "method_name":   "A1_PCR",
    "run_name":      "wt2_seed42",
    "seed":          42,
    "results_root":  "./results",

    "model_name":    "gpt2",
    "tokenizer_name": "gpt2",

    "dataset":       "wikitext2",
    "max_seq_len":   512,
    "batch_size":    16,
    "num_workers":   2,

    "learning_rate": 5e-5,
    "num_epochs":    3,
    "warmup_steps":  100,
    "grad_clip":     1.0,
    "weight_decay":  0.01,

    "limit_train_batches": 100,
    "limit_eval_batches":  50,

    "log_every_n_steps":        25,
    "save_checkpoints":         False,
    "checkpoint_every_n_steps": 500,
    "keep_last_n_checkpoints":  3,

    # ─ Method-specific (PCR)
    "pcr_predictor_dim":    256,            # predictor MLP gizli boyut
    "pcr_predictor_layers": 2,              # MLP kat sayısı
    "pcr_delta":            1e-6,           # sayısal stabilite
    "pcr_temperature":      1.0,            # softmax sıcaklığı
}

In [ ]:
# ─── HÜCRE 3: Imports ──────────────────────────────────────────────────
import math, time, json
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from arf_lib import (
    seed_everything, get_device, gpu_info, gpu_peak_mb,
    GPT2Wrapper, BaseRouterAttention,
    StandaloneTrainer, load_text_dataloaders,
    run_pipeline, save_results, append_to_index, make_plots,
)

seed_everything(PARAMS["seed"])
print(f"Seed set: {PARAMS['seed']}")
print(f"Device: {get_device()}")

In [ ]:
# ─── HÜCRE 4: Method — Predictive Coding Router ────────────────────────
class PredictiveCodingRouter(BaseRouterAttention):
    """Score = 1 / (prediction_error + δ).
    Tek predictor MLP tüm head'lere broadcast olur (prototip)."""

    def __init__(self, gpt2_config, method_params):
        super().__init__(gpt2_config, method_params)
        mp = method_params
        pred_dim = int(mp.get("pcr_predictor_dim", 256))
        n_layers = int(mp.get("pcr_predictor_layers", 2))
        self.delta = float(mp.get("pcr_delta", 1e-6))
        self.temperature = float(mp.get("pcr_temperature", 1.0))

        layers = []
        in_d = self.embed_dim
        for _ in range(max(1, n_layers - 1)):
            layers += [nn.Linear(in_d, pred_dim), nn.ReLU()]
            in_d = pred_dim
        layers.append(nn.Linear(in_d, self.embed_dim))
        self.predictor = nn.Sequential(*layers)

    def _prediction_error(self, hidden):
        # Memory-efficient pairwise MSE:
        # ||x_j - pred_i||^2 = ||pred_i||^2 - 2 * pred_i . x_j + ||x_j||^2
        # 4D intermediate tensor olusturmaz -> O(B*S^2) bellek, O(B*S^2*H) degil.
        pred = self.predictor(hidden)                                 # [B,S,H]
        x_sq = (hidden * hidden).sum(-1)                              # [B,S_j]
        p_sq = (pred   * pred  ).sum(-1)                              # [B,S_i]
        cross = torch.matmul(pred, hidden.transpose(-1, -2))          # [B,S_i,S_j]
        errors = p_sq.unsqueeze(2) - 2.0 * cross + x_sq.unsqueeze(1)  # [B,S_i,S_j]
        return errors.clamp(min=0.0)                                  # numerik temizlik

    def _attend(self, q, k, v, hidden_states, attention_mask):
        errors = self._prediction_error(hidden_states)
        scores = 1.0 / (errors + self.delta)
        scores = scores / self.temperature
        scores = scores.unsqueeze(1).expand(-1, self.num_heads, -1, -1)

        causal = self._causal_mask(q.size(-2), k.size(-2), q.device)
        scores = scores.masked_fill(~causal, float("-inf"))
        if attention_mask is not None:
            scores = scores + attention_mask

        weights = F.softmax(scores, dim=-1)
        weights = self.attn_dropout(weights)
        out = torch.matmul(weights, v)
        self._last_stats = {"mean_err": float(errors.mean().item())}
        return out, weights

print("PredictiveCodingRouter tanımlandı.")

In [ ]:
# ─── HÜCRE 5: Build model & adapter ────────────────────────────────────
def build_model_and_adapter(params):
    wrapper = GPT2Wrapper(params["model_name"])
    wrapper.inject_attention(PredictiveCodingRouter, params)
    n = sum(p.numel() for p in wrapper.parameters())
    print(f"PCR injected → toplam param: {n:,}")
    return wrapper, None

In [ ]:
# ─── HÜCRE 6: Run ──────────────────────────────────────────────────────
# Tek satır: model + opsiyonel adapter inşa et, pipeline çalıştır
model, adapter = build_model_and_adapter(PARAMS)
run_dir, result = run_pipeline(model, PARAMS, adapter=adapter)
fm = result["final_metrics"]
print("\n" + "=" * 60)
print(f"RUN DIR: {run_dir}")
print(f"Final test PPL:             {fm['test_ppl']:.4f}")
print(f"Final test BPC (per-char):  {fm['test_bpc']:.4f}")
print(f"Final test bits-per-token:  {fm['test_bits_per_token']:.4f}")
print(f"chars/token:                {fm['chars_per_token']:.3f}")
print(f"Duration: {result['duration_seconds']:.1f}s")
print("=" * 60)

In [ ]:
# ─── HÜCRE 7: Display plots (inline) ───────────────────────────────────
from IPython.display import Image, display
import os
plots_dir = run_dir / "plots"
if plots_dir.exists():
    for p in sorted(plots_dir.glob("*.png")):
        print(f"📈 {p.name}")
        display(Image(filename=str(p)))
else:
    print("Plots not generated.")
print(f"\nAll outputs in: {run_dir}")
print(f"Index file: {Path(PARAMS['results_root']) / '_index.csv'}")